#  <center> Problem Set 1 (Perovskites) <center>
<center> Spring 2026 <center>
<center> 3.C01/3.C51, 7.C01/7.C51, 10.C01/10.C51, 20.C01/20.C51 <center>
<center> Due: Monday, April 6, 2026 at 3:00 PM ET. <center>

<b>Name:</b>

<b>Kerberos ID:</b>

# Overview:
In this PSET, you will:

* Learn the basics of processing your data and training a machine learning model with PyTorch:
    * Processing data, including exploring OHE vs. other featurization techniques
    * Formatting your data into a `Dataset` object and wrapping in a `DataLoader` instance
    * Implementation of a Multi-Layer Perception in `PyTorch`
    * Setting up a training and testing loop, and how to do evaluation
    * Using a GPU to accelerate training!
    * Find the best hyperparameters


* Learn how to build some simple architectures for **classification** and **regression**:
    * Train a logistic regression model with `scikit-learn` to _classify_ breast cancers based on metabolite abundance
    *  Train a random forest classifier on the same data
    * Train a MLP using `PyTorch` on the same data
    * Train a MLP with `PyTorch` to _regress_ perovskite hull energies from their compositions
    * Apply regularization techniques to avoid overfitting (L1 and L2)
    * Apply physical descriptor-based encoding to improve training performance


## Instructions

- This problem set has two modeling tasks with several sub-questions. Some are marked grad version, which are required for graduate students (X.C51) but optional for others. Points for all students are in <span style="color:blue">blue</span>, while grad-only points are in <span style="color:orange">orange</span>. There is one problem that is undergrad only in <span style="color:purple">purple</span>. The total points are 75 for undergraduates and 100 for graduates.
  
- To get started, make your own copy of this notebook template in Colab (e.g., “Save a copy in Drive”) before editing.

    - Important: this problem set requires a GPU. In Google Colab go to `Edit -> Notebook settings` and set the `Hardware accelerator` to a GPU before running the notebook (changing the runtime resets the notebook). See the GPU section below for additional help.

- Collaboration is encouraged and AI tools are permitted, but submitting work that is not your own is plagiarism. Any collaboration or assistance from others or from an LLM (including utilities integrated in Colab) must be described at the end of your submission.

- Additional notes about how to use this template:
    - Put your code in the code blocks flagged with `############# Code ##########`.

    -  Numerical answers yielded from running the code should be included in an Answer Block (see next cell). 

    - We have provided print statements where numerical answers are expected.

    -  Your answer should be contained in a variable which you defined either in the Answer Block or the Code Block.

    - When a qualitative answer is expected, place those comments as Markdown/Text cells; when asked for within Code blocks, you can write answer as code comments by placing a # before your answer.

- Submission: upload your completed `pset1.ipynb` to Gradescope. Ensure the notebook runs without error and includes all necessary code, plots, and outputs. Comments are encouraged; place conceptual answers in Markdown/Text cells.

In [ ]:
# example answer block
########## Answer ############

ans = 2
print("My answer is: {}.".format(ans))

# My regressor over-fitted the training data, I need to add regularization

########## Answer ############

## Imports

In [ ]:
# import packages
import numpy as np
import sklearn
import pandas as pd
from sklearn.model_selection import train_test_split

# models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from torch.utils.data import Dataset, DataLoader
from sklearn import preprocessing
from torch import nn
import torch.nn.functional as F

# metrics
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.simplefilter("ignore", category=ConvergenceWarning)
import torch
from tqdm import tqdm


# plotting style, you can choose your own parameters
import matplotlib

matplotlib.rcParams.update({'font.size': 15})
matplotlib.rc('lines', linewidth=3, color='g')
matplotlib.rcParams['axes.linewidth'] = 2.0
matplotlib.rcParams['axes.linewidth'] = 2.0
matplotlib.rcParams["xtick.major.size"] = 6
matplotlib.rcParams["ytick.major.size"] = 6
matplotlib.rcParams["ytick.major.width"] = 2
matplotlib.rcParams["xtick.major.width"] = 2
matplotlib.rcParams['text.usetex'] = False

In [ ]:
# A helper function for students to produce plots
def plot_clf(model, X, y, title):

    '''
        A function to plot confusion matrix and ROC curve

        Args:
            model(classifier object): model object (e.g. RandomForestClassifier, LogisticRegression)
            X(np.array): feature set
            y(np.array): label set
            title(str): plot name

        Example Usage:
            plot_clf(model, X_test, y_test, "test")
    '''

    fig, [ax_roc, ax_conf] = plt.subplots(1, 2, figsize=(12, 6))
    fig.tight_layout()

    RocCurveDisplay.from_estimator(model, X, y, ax=ax_roc)
    ConfusionMatrixDisplay.from_estimator(model, X, y, ax=ax_conf)

    ax_roc.set_title('{} ROC'.format(title))
    ax_conf.set_title('{} Confusion Matrix'.format(title))

    plt.show()

## Grading guideline

- Didn't answer the question 0%
- Showed some attempts, but clearly didn't try enough: 25%
- Showed solid attempts (showed code) but does not answer the question directly: 50%
- Showed solid attempts and get the question wrong: 60-80%
- Showed solid attempts with some small mistakes: 80-90%
- Showed code and answered the questions correctly: 100%

## Download required data

In [ ]:
# for 1st set of tasks: binary classification
# TODO: update the datapaths
! wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps1-nonbio/data/breastcancer_X.csv
! wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps1-nonbio/data/breastcancer_y.csv

# for 2nd set of tasks: regression of perovskite binding energies

! wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps1-nonbio/data/data_perov/mendeleev.csv
! wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps1-nonbio/data/data_perov/perov_train.csv
! wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps1-nonbio/data/data_perov/perov_val.csv
! wget https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps1-nonbio/data/data_perov/elements.npy

# Problem 1: Breast cancer classification from metabolite data

## Background

Imagine if all diseases could be diagnosed with from a tiny drop of blood. While that day may still be far off, much progress has been made in searching for what are called *biomarkers*, molecules in the blood that are associated with particular diseases. Many studies use [mass spectrometry](https://en.wikipedia.org/wiki/Mass_spectrometry) to search for such diagnostic molecules. Biomarkers can be proteins, nucleic acids, lipids, or any of thousands of other chemical compounds that are found in the blood (see Figure 1).

<div align="center">
  <img src="https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps1-nonbio/figures/metabolites.png" width="600px" />
</div>
<div align="center">Figure adapted from (<a href="https://doi.org/10.1016/j.jpba.2016.07.037">Gonzalez-Riano et al.</a>)</div>

Because many of these chemical compounds are products of metabolism, they are often called *metabolites*, and the detection of metabolites is called [metabolomics](https://en.wikipedia.org/wiki/Metabolomics). Recent studies have shown that metabolites can be predictive of human health conditions ((Bar et al. [link](https://doi.org/10.1038/s41586-020-2899-z)), (Evans et al. [link](https://doi.org/10.1038/s41598-020-74925-y))). In this problem, we will use two different classification methods to detect breast cancer from patients' metabolite data collected from human plasma/serum, following data processing steps found [here](https://www.metabolomicsworkbench.org/data/DRCCMetadata.php?Mode=Study&StudyID=ST000355) (Huang et al. [link](https://doi.org/10.1186/s13073-016-0299-4)).

## 1.1 <span style="color:blue">(5 points) </span>  Load and inspect the raw data

To perform supervised machine learning on vector-valued data, you need labeled examples $(\mathbf{x},y)$, where $\mathbf{x}$ is a vector of input features and $y$ the known label. Your goal is to train a model $\hat{f}$ that maps features to labels: $\hat{f}(\mathbf{x}) \approx y$. For diagnosing breast cancer from metabolite data, the metabolite signal is $\mathbf{x}$, and the binary label (positive or negative) is $y$, both provided as `.csv` files.

**Task 1:** We provide code utilizing pandas and numpy to load the data. Make sure you understand what each line of code is doing. **Briefly explain each line** by providing short comments below

**Task 2:** Use `X.shape` to report the number of samples and features per sample.

> **Note**: You will have to do it by yourself again in Problem 2.

In [ ]:
p1_X = pd.read_csv("./breastcancer_X.csv", header='infer', index_col=0) # Explain me
p1_y = pd.read_csv("./breastcancer_y.csv", header='infer', index_col=0) # Explain me

metabolite_name = p1_X.columns.tolist() # Explain me

p1_X = p1_X.values # Explain me
p1_y = p1_y.values # Explain me

Report how many examples are in this dataset and the number of features for each data point.

In [ ]:
########## Answer ############

print("There are {} samples.".format(N_samples))
print("There are {} features per sample.".format(N_features))

########## Answer ############

## 1.2 <span style="color:blue">(5 points) </span>  Generate train/test splits.

To fairly evaluate the performance, split your data into a training data set and testing data set. Only training data should be used to train the model; testing data are for unbiased evaluations of model performance. During training, the model should not have access to *any* information about the testing dataset, so you should not train (or preprocess!) on the testing data. Use [sklearn.model_selection.train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) to *randomly* split your dataset into training and testing data with an **80%:20% ratio** (you should get two feature arrays, and two corresponding label arrays). 



**Task:** Show your `train_test_split` code, print the shapes of your four variables, `X_train`, `X_test`, `y_train`, and `y_test` and ensure that the dimensions match your expectations.

In [ ]:
########### Code #############



########### Code #############

In [ ]:
########## Answer ############

X_train_shape = None # fill me in
y_train_shape = None # fill me in
X_test_shape = None # fill me in
y_test_shape = None # fill me in


print("X_train shape: {}".format(X_train_shape))
print("y_train shape: {}".format(y_train_shape))

print("X_test shape: {}".format(X_test_shape))
print("y_test shape: {}".format(y_test_shape))

########## Answer ############

## 1.3 <span style="color:blue">(5 points) </span> Preprocess the data through scaling

Features in your dataset may have different units and magnitudes: for example, an atom's molecular weight (1-294 grams/mol) and covalent radius ($30-250 \times 10^{-12}$ meters) have different ranges. To avoid unfairly weighting features, we *standardize* all features to a consistent scale.



**Standardization:** Let $N$ be the total number of features, and $M$ is the total sample size. Let $\{ X_{0, j} ... X_{i, j}... X_{N-1, j} \} $ be the feature vector for the $j^\text{th}$ sample (patient), where $X_{i, j}$ refers to the unnormalized abundance of the $i^\text{th}$ metabolite in the $j^\text{th}$ patient. The mean and standard deviation (scikit-learn and numpy use a normalization of $M$ for the *population* standard deviation, rather than $M-1$ for the sample standard deviation, which is numpy's default as well. pandas defaults to the *sample* standard deviation) of each feature are calculated as:

$$\begin{aligned}
    \mu_i &= \frac{1}{M} \sum_{j=1}^{M} X_{i, j} \quad & \sigma^2_{i} &= \frac{1}{M} \sum_{j=1}^{M} (X_{i, j} - \mu_i)^2
\end{aligned}$$

Each feature is transformed under an affine mapping for the feature vectors $X_{i, j}$. We call the transformed feature vectors $X'_{i, j}$:

$$X'_{i, j} = \frac{1}{\sigma_i} (X_{i, j} - \mu_i)$$

Note that this procedure is invertible (meaning you can get the original feature vector back if you know $\sigma_i$ and $\mu_i$), meaning no information loss for your data. 

**Task:** Use scikit-learn's [preprocessing.StandardScaler](https://scikit-learn.org/stable/modules/preprocessing.html#standardization-or-mean-removal-and-variance-scaling) to process your input features, `X`, into `X_train_scaled`, following the feature standardization above. Show the $\mu_i$ and $\sigma_i$ of each feature are 0 and 1 respectively after scaling (use `np.mean` and `np.var`). Apply the same ScalerTransform to `X_test` to produce `X_test_scaled`. Note: the ScalerTransform should *only* be fit upon `X_train` and applied to `X_test`. Take one sentence to discuss why, and what information leak might occur if one were to fit the scaler upon both `X_train` **and** `X_test`.

Scale the datasets

In [ ]:
########### Code #############



########### Code #############

Print the mean/variance for each transformed feature.

In [ ]:
########## Answer ############

train_mean = None # fill me in
train_variance = None # fill me in

test_mean = None # fill me in
test_variance = None # fill me in

print("The means of the transformed feature train set are {}".format(train_mean) )
print("The variances of the transformed feature train set are {}".format(train_variance) )
print("The means of the transformed feature test set are {}".format(test_mean) )
print("The variances of the transformed feature test set are {}".format(test_variance) )
########## Answer ############

**Question**: In 1-3 sentences, describe why the ScalerTransform should _only_ be fit to `X_train` and applied to `X_test`; what information leak might occur if one were to fit the scaler on the entire dataset prior to splitting?

**Write answer here**

## 1.4 <span style="color:blue">(10 points) </span> Training a logistic regression classifier with `scikit-learn`

Now, you should be ready to train a logistic regression model and evaluate its performance on the test data. We will use simple model modules from scikit-learn, a machine learning library. We will use three handy evaluations for this task:



* **Confusion matrices** [link](https://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html): A visualization to stratify model performance by looking at positive and negative samples separately, and how many are correctly classified (true positives and negatives) vs. misclassified (false positives and negatives).



* **ROC Curve** [link](https://en.wikipedia.org/wiki/Receiver_operating_characteristic): The receiver operating characteristic (ROC) curve plots the true positive rate (TPR) against the false positive rate (FPR) at different decision thresholds. We usually report the area under the curve (AUC) of the ROC (AUC-ROC) for a scalar metric.



* **Precision-recall (PR) curve** [link](https://en.wikipedia.org/wiki/Precision_and_recall): This plots the precision (proportion of true positives to all predicted positives) against the recall (or true positive rate). We report the area under the PR curve (AUPRC) as a single metric from the PR curve. This evaluation is handy when the dataset is highly imbalanced, i.e. many more positives than negatives or vice versa.

**Task**: Train a logistic regression model on the scaled training data and evaluate it on testing data with scikit-learn's [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) class. For both the train and test datasets, generate plots for confusion matrices and the ROC curve with help of `plot_clf`, and report the AUC-ROC score. Finally, plot a histogram with [matplotlib.pyplot.hist](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.hist.html) of the model coefficients (you can retrieve model coefficients from `model.coef_`, where `model` is your model) to get an understanding of the parameters learned for the dataset's features.

In [ ]:
########### Code #############


########### Code #############

Report the AUC for both the train and test datasets.

In [ ]:
########## Answer ############

print("The training AUC score is {:.3f}".format(train_auc) )
print("The testing AUC score is {:.3f}".format(test_auc) )

########## Answer ############

Generate plots for the confusion matrices and the ROC curve for both training and testing. Please use the `plot_clf` function defined above.

In [ ]:
########### Code #############


########### Code #############

In [ ]:
########### Code #############


########### Code #############

## 1.5 <span style="color:blue">(5 points) </span> Introduce L1 regularization

In the previous part, you visualized the distribution of model coefficients. Now, we consider how these change after regularizing the model with L1 loss (we'll explore L2 regularization in Part 2). 



**Task 1:** For this part, modify your logistic regression model to include L1 regularization using the keyword arguments `penalty='l1'`, `solver='saga'`. Use the same pre-processed training and testing data from section 1.1. Print the AUC-ROC score, plot the ROC curve, and plot the confusion matrix for both training and testing data. 

In [ ]:
########### Code #############


########### Code #############

Report the ROC-AUC score.

In [ ]:
########## Answer ############

print("The training AUC score is {:.2f}".format(train_auc) )
print("The testing AUC score is {:.2f}".format(test_auc) )

########## Answer ############

Correspondingly, generate the new confusion matrix and ROC curve.

In [ ]:
########### Code #############



########### Code #############

**Task 2:** Probe the effect of regularization by plotting the histogram of the new model coefficients to find any qualitative changes. Comment on any differences between the two models' distribution of model coefficients, thinking specifically about the *geometric* interpretation of L1 regularization.

In [ ]:
########### Code #############


########### Code #############

Comment on the histogram you obtained, by comparing it to the one generated from the unregularized model's coefficients.

**Write your answer here**

## 1.6  <span style="color:blue">(optional +2.5 points) </span>  Connect model coefficients back to metabolites

**Task:** The column of your feature set `X` which you loaded in section 1.1 contains the chemical name for each metabolite. Based on the trained model from section 1.4, identify the top 5 metabolites that are most correlated the most with a positive diagnosis.

Code to identify the top 5 metabolites that positively correlated the most with positive diagnosis.

In [ ]:
########### Code #############


########### Code #############

Report the metabolites you identified.

In [ ]:
########## Answer ############

print("The top 5 metabolites are {}".format(", ".join(metabolites)) )

########## Answer ############


The chemicals found do not have to be the same. The students need to show that they know how to retrieve model coefficients and sort based on the values.

## 1.7 <span style="color:blue">(5 points) </span>  Hyperparameter tuning the regularization parameter

To optimize your model further, one may tune the hyperparameters, which are parameters whose values are used to control some aspect of the model or the learning process, but is not optimized during the training. These might include the layer widths, the number of layers, the learning rate, optimizer, and more. This is done for a number of reasons: to prevent overfitting, make training most efficient, improve generalization, etc.

**Task:** For this problem, tune the regularization parameter by testing 4 provided values, and selecting the best model based on the training set (or you may create a separate validation set, if desired) to select the parameters. Report the hyperparameter with the best train performance, and that model's performance on the test set.

Scan over the following regularization values `C=[0.01, 1, 5, 10]` (in `scikit-learn`, the regularization parameter `C` is inversely related to the regularization strength) and report which one yields the best performance on the train dataset. What is its performance on the test data?

In [ ]:
########### Code #############

best_value = None
best_metric = 0
C_values = [0.01, 1, 5, 10]

########### Code #############

In [ ]:
########### Answer #############

print(f"Best value: C = {best_value}")
# Performance on test AUC:
print("The hyperparameterized model's test AUC score is {:.2f}".format(test_auc) )

########### Answer #############

**Write answer here**

## 1.8 <span style="color:blue">(5 points) </span>  Training a random forest classifier with `scikit-learn`

Besides Logistic Regression classifiers, another popular classification model architecture is Random Forests, which are ensembles of decision tree classifiers. Ensemble models can be often useful for mitigating overfitting and reducing the variance of your classification method. 

To ensure our model's performance isn’t just due to luck, we evaluate it on different splits of the data, known as *k-fold cross validation*. Following Figure 1, we effectively run the same experiment $K$ times, each time using a distinct train/validation split. This yields $K$ different performance values, which can be averaged to yield a more robust reflection of performance.

<div align="center">
  <img src="https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps1-nonbio/figures/crossval.png" width="600px" />
</div>
<div align="center">Overview of k-fold cross-validation. Source: <a href="https://scikit-learn.org/stable/modules/cross_validation.html">link</a></div>

**Task:** Initialize a Random Forest classifier using `RandomForestClassifier` from the scikit-learn library. Use the following parameter set: `{max_depth=2, n_estimators=20}`. Use the function [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html#sklearn.model_selection.cross_val_score) to perform a 5-fold cross validation, using `scoring='roc_auc'` as the metric to report; report AUC-ROC scores via their mean and standard deviation.

To relearn the scaler transform in each fold, `sklearn` has the [Pipeline](https://scikit-learn.org/stable/modules/compose.html#) API to chain together multiple steps in a machine learning model. Feed `cross_val_score` a `Pipeline` so that it correctly fits the scaler to only the training set within each cross-validation fold.

(to minimize the variance of our method to reduce error arising from epistemic error), we may often look to train multiple copies of a machine learning model, i.e., an _ensemble_ machine learning method.

Here, train a Random Forest classifier, which is an ensemble of decision trees. Random Forests, empirically, work really well on tabular data, but can be known to overfit easily. To mitigate this concern, also perform cross validation.

In [ ]:
########### Code #############
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score


scaler = # fill this in (reusing your scaler implementation from 1.3)
model = # fill this in 
pipeline = Pipeline([('scaler', scaler), ('model', model)])

# Now call cross_val_score() and feed it: your pipeline as the estimator 
# and your data (p1_X & p1_y) letting cross_val_score() handle the test / train split for you.

########### Code #############

Report the cross-validated ROC-AUC score.

In [ ]:
########## Answer ############

print("The mean of CV scores is {:.2f}".format(mean) )
print("The std of CV scores is {:.2f}".format(std) )

########## Answer ############

# Problem 2: Training a Multi-Layer Perceptron (MLP) to predict binding affinities

## Background
This problem covers neural networks to predict the $E_{hull}$ of crystalline perovskite materials with varying composition; predicting these properties lets us design new materials with desired function. 

### Structures and properties of perovskites

Natural perovskite is a mineral composed of calcium titanium oxide ($\text{CaTiO}_3$). It was discovered in Russia by Gustav Rose and named after mineralogist L. A. Perovski. Gernally, any material whose crystalline structure is similar to the perovskite mineral is a *perovskite* (if you're curious, we encourage you to learn more about crystal structure [here](https://ocw.mit.edu/courses/res-3-004-visualizing-materials-science-fall-2017/pages/student-projects-by-year/epfl2017/a-basic-and-fun-introduction-to-crystalline-structures/). Because their structure can support many possible element compositions, perovskites are a quite abundant structural family, and have wide-ranging properties, applications, and importance, such as in solar cells, piezoelectrics or catalysts.

In perovskites, A is typically a large cation (alkali metal, rare earth, or organic like methylammonium), B is a transition metal, and X is an anion (oxide, fluoride, nitride, or halide) bonded to both cations. The ideal cubic structure (space group $Pm\bar{3}m$), has B in 6-fold octahedral coordination and A in 12-fold cuboctahedral coordination. However, real materials often deviate from this structure, reducing symmetry. These distortions, such as in $\text{BaTiO}_3$, lead to properties like piezoelectricity. Many elements can form stable perovskite structures, allowing for a vast possible set of compositions. 

Because their compositions and thus their properties are extremely tunable, pervoskites are exciting materials for many applications. For example, take solar cells based on organic-inorganic halide perovskites (with an organic cation as A site; lead or a related element as B site; and Cl, Br or I as X): since their discovery about a decade ago, their efficiency has approached that of silicon photovoltaics (around 25%), yet are much cheaper to manufacture and can be made into thin films that are flexible and foldable. Despite their promise, though, perovskite solar cells suffer from stability issues that prevent mass adoption. 



<div align="center">
  <img src="https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps1-nonbio/figures/perov.png" width="400px" />
</div>
<div align="center">Perovskite crystal structure</div>



<div align="center">
  <img src="https://raw.githubusercontent.com/coleygroup/ML4MolEng/main/psets/ps1-nonbio/figures/application.png" width="400px" />
</div>
<div align="center">Applications for molecular sensing</div>

<div align="center">Perovskite structure and applications (Chen et al. <a href="https://doi.org/10.1039/C8RA01155E">link</a>)</div>

### Data Generation and Processing

The data you will be modeling is generated with high-quality Quantum Chemistry calculations using Density Functional Theory (DFT). Each structure takes hundreds of CPU hours to calculate, and thus fairly expensive to obtain. In this question, you are asked to predict $E_{hull}$ (Energy above hull). It has the unit of eV/Atom. It describes the thermodynamic stability of the crystal structure. If $E_{hull} = 0.0$, it means the structure is at its most stable form. Building these predictive models can help theorists reduce their computational cost when designing for more stable materials. 

**Energy Above Hull:** The energy of decomposition of this material into the set of most stable materials at this chemical composition, in eV/atom. Stability is tested against all potential chemical combinations that yield that composition. For example, a $\text{Co}_2\text{O}_3$ structure (cobalt oxide) would be tested for decomposition against other $\text{Co}_2\text{O}_3$ structures, against Co and $\text{O}_2$ mixtures, and against CoO and $\text{O}_2$ mixtures. $E_{hull}$ is computed as: 

$$E_{\text{Co}_2\text{O}_3} - \min( 2 E_{\text{Co}} + \frac{3}{2} E_{\text{O}_2} ,  2E_{\text{CoO}} + E_{\text{O}_2} ,   E_{\text{Co}_2\text{O}_3} )$$

Let's start with loading the dataset! You will be asked to use Pandas DataFrames. If you have not used Pandas before, please work through this [tutorial](https://pandas.pydata.org/pandas-docs/stable/user_guide/10min.html).

## 2.1: <span style="color:blue">(5 points) </span> Encoding the data

For this second task, we will utilize perovskite data to predict their $E_{hull}$ values. Before we can build a model, we need to process the data by one-hot encoding the perovskites based on their elements.

After loading the pandas dataframe, take a moment to inspect your data by looking at the rows (samples) and columns (features). The important chemical information is stored in columns `['A', 'B', 'X']` and  is stored in the column `e_above_hull`.

First, you will train a model that uses chemical compositions (e.g. ) to predict Energy above Hull () which is a scalar property. How can you map a chemical formula like  to numeric features for training a model? You need to define a dictionary to encode elements into numerical features so that we can apply programs like linear regressions on these input features. One way to do is to use one-hot encoding to transform elements into bit vectors. For example, if we have elements Fe, Ni, and Mn for A and B sites, we can assign bit vectors of size 3 to fully encode this label information. We do not encode the elements for the X site because Oxygen is the only element available in our dataset for X. We present simple examples on how you should encode perovskite chemistries in Table 1.

<div align="center">Table 1: Example bit vector representation of A/B sites in perovskites</div>


| A site | B site | bit vector representation |
| --- | --- | --- |
| 'Fe' | 'Ni' |  |
| 'Mn' | 'Ni' |  |
| 'Fe' | 'Mn' |  |
| 'Fe' | 'Fe' |  |


**Task:** After reading the documentation for [preprocessing.LabelBinarizer()](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelBinarizer.html), use it to transform the features `['A', 'B']` into one-hot encoded arrays with the alphabet for A/B sites from `elements.npy`. Next, convert the hull energies to a `np.array` for use in regression. You'll have a separate test set that you should should process separately, and not combine or split further for the first parts of this problem. Your code should output `X_train` and `y_train`, and `X_test` and `y_test`. You will randomly split the first two arrays, `X_train` and `y_train`, into train/validation with an 80%/20% ratio in a later cell when DataLoaders are made.

> **Hint:** There are many ways to do it. One way to do it is to loop over rows (perovskites) in your dataframe and encode A and B into bit vectors separately, and then stack the two arrays horizontally using `np.hstack`.

Generate the X matrix and y vector from processing `perov_train` and `perov_test` appropriately.

> **Hint**: Note that the one-hot encoding we perform should still give us a 2D matrix of `n_samples` x `n_features`, but now `n_features` will be different from the number of features originally.

In [ ]:
perov_train = pd.read_csv("perov_train.csv") # read train
perov_test = pd.read_csv("perov_val.csv") # read test
all_elements = np.load('./elements.npy', allow_pickle=True) # Read all elements


# Your code to featurize elements
########### Code #############


########### Code #############

Report the number of samples, number of features, and the number of possible values for one (not yet one-hot-encoded) feature you might have.

In [ ]:
########## Answer ############

print("There are {} samples.".format(N_samples))
print("There are {} features per sample.".format(N_features))
print("There are {} possible values for one feature.".format(N_feat_vals))

########## Answer ############

## Accelerating neural networks with GPUs

Scikit-learn provides common machine learning models but is limited in customization and speed of training. In contrast, PyTorch (along with TensorFlow, JAX, and CNTK) is designed for deep learning, allowing manual expressivity and efficient training through usage of GPU speedup. PyTorch uses reverse-mode automatic differentiation (AD) to compute gradients automatically, making it easier to optimize model weights. It also supports training on a Graphical Processing Unit (GPU), which enables significantly accelerated training.

In this part, you'll call certain functions from PyTorch to construct a Multi-Layer Perceptron, which is composed of alternating linear affine and non-linear transformations (layers), and train it on a GPU in Google Colab (where PyTorch is pre-installed). After this exercise, we hope you will be comfortable building your own machine learning workflow using PyTorch by adapting this and other example code.

**Optional reading** The [PyTorch Tutorial](https://pytorch.org/tutorials/beginner/basics/intro.html) and [Quickstart Guide](https://pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html) are great companion resources to the functional PyTorch primer we present in this question. Likewise for debugging, we suggest leafing through [this guide](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/guide3/Debugging_PyTorch.html).

## PyTorch setup: GPU basics, Datasets and DataLoaders

There are no points associated with this part, but spend some time understanding the cells in this section. 

**Setting up GPU usage:** In Google Colab, go to `Edit` > `Notebook Settings`, and select T4 GPU under `Hardware accelerator`. You can also do this via `Runtime` > `Change runtime type` or by clicking the three dots in the top right corner. Verify GPU access using the provided code. In PyTorch, the main object is `torch.Tensor`, similar to `np.array`. In the second cell, confirm that you can move the sample tensor made to and from the GPU, with `.to(device).`



**Building `Datasets` and `DataLoaders` in PyTorch:** To organize our data, PyTorch wisely modularizes the data into smaller chunks, or minibatches, that allow the model to process one at a time to avoid memory overhead. In PyTorch, data is stored in `torch.utils.data.Dataset` and batched with `torch.utils.data.DataLoader`.



From section 2.1, you should already have the data featurized for the train and test datasets (with the latter being loaded from the test file), so we split the `X_train` and `y_train` data into a train and validation set for you. Take a moment to parse the `PerovskiteDataset` class construction that implements `Dataset` for this problem. Also parse the `DataLoader` instances that wrap each `PerovskiteDataset` object made, paying attention to the `shuffle=True` parameter, to avoid overfitting to minibatches; and the size of the minibatch, set with `batch_size=128`. You will need to know how to construct your own in PSET 2.

## Check GPU usage

In [ ]:
# Check if your GPU is requested successfully or not
assert torch.cuda.device_count() != 0

To work with GPU-accelerated training, we need to use `PyTorch`'s `Tensor` objects, which can help us manage CPU vs. GPU usage.

Demonstrate moving this sample tensor to and from the GPU.

In [ ]:
numpy_sample = np.zeros((3, 5))
tensor_sample = torch.Tensor(numpy_sample)
print(tensor_sample.device)

### Your Code Here
tensor_sample = tensor_sample.to('cuda')
print(tensor_sample.device)
tensor_sample = tensor_sample.to('cpu')
print(tensor_sample.device)


###

## Build Datasets and DataLoaders in PyTorch

Below, we provide you with an example of a `Dataset` instance, the `SequenceDataset` class, to format your data.

Note that tensors are not moved onto the GPU at initialization or inside `__getitem__`; GPU memory is typically far more limited than CPU-available memory. To prevent out-of-memory errors, we typically minimize the amount of data on the GPU at any time. During `.forward()`, the easiest way to do this is by only putting one batch at a time on the GPU.

In [ ]:
# Generate dataset
class PerovskiteDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.Tensor(np.array(X))  # store X as a pytorch Tensor
        self.y = torch.Tensor(np.array(y))  # store y as a pytorch Tensor
        self.len=len(self.X)                # number of samples in the data

    def __getitem__(self, index):
        return self.X[index], self.y[index] # get the appropriate item

    def __len__(self):
        return self.len

Use the above `Dataset` object to make train, validation, and test `PervoskiteDataset` objects, then wrap each in a `DataLoader` object. Here are some handy arguments you can use to tweak the DataLoader object:
* `batch_size`: the number of examples that your model will see during one forward/backward call.
* `shuffle`: whether or not to shuffle your data between epochs; if it is set to True, then each epoch's batches will be different in identity.
* `num_workers`: useful for when you have a lot of data to load in when making a batch; this allows you to multiprocess batch formation before they are executed on the GPU. it won't make much difference in this homework, but can be handy down the line :)

In [ ]:
########### Code #############
X_subtrain, X_val, y_subtrain, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0)

train_data = PerovskiteDataset(X_subtrain, y_subtrain)
val_data = PerovskiteDataset(X_val, y_val)
test_data = PerovskiteDataset(X_test, y_test)

batch_size = 128
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=True)

########### Code #############

Run this cell to check that your DataLoaders work as expected.

In [ ]:
########### Code #############

for loader in [train_dataloader, val_dataloader, test_dataloader]:
    for index, batch in enumerate(loader):
        # Your batch returns a X, y stacked in a batch
        X_batch, y_batch = batch[0], batch[1]
        print(X_batch.shape, y_batch.shape)
    print()

########### Code #############

There should be 11 training set batches, 3 validation batches, and 2 test batches, with sizes shown above. Each set has an final, impartial batch. 

## 2.2 <span style="color:blue">(10 points) </span>   Define the MLP in PyTorch

We will now proceed to build a MLP using PyTorch’s `nn.Module` class. The `nn.Module` class requires two methods: an `__init__` method to define components and layers, and a `forward` class where you define the network's computation, i.e. how to compute a prediction with the layers given input data. Many standard layers, such as `nn.Linear` affine layers and either the `nn.ReLU` or `nn.Tanh` layer for your activation function, are already made in PyTorch. You can use a `nn.Sequential` module to stack layers to automatically process your data in order. We ensure all layers are class attributes so PyTorch can handle auto-differentiation properly.



**Task:** We have provided the skeleton code for implementing your MLP in PyTorch as a `nn.Module`. Fill in the remainder of the code. Use three hidden layers, of widths 256, 256, 256; and ReLu activation layers after each `nn.Linear` layer.

Look at the following code snippet to understand how the linear layer works in PyTorch. Take careful note of the dimensions of the input and output.

In [ ]:
linear = torch.nn.Linear(2, 3)

input_tensor = torch.ones((4, 2))
output_tensor = linear(input_tensor)

print(input_tensor, output_tensor, input_tensor.shape, output_tensor.shape)

Look at the following code snippet to understand how the ReLU layer works in PyTorch (the Tanh layer is similar). Take careful note of the dimensions of the input and output.

In [ ]:
relu = torch.nn.ReLU()

input_tensor = torch.ones((4, 2))
output_tensor = relu(input_tensor)

print(input_tensor, output_tensor, input_tensor.shape, output_tensor.shape)

Look at the following code snippet to understand how to stack layers with the Sequential module.

In [ ]:
layer1 = torch.nn.Linear(2, 3)
layer2 = torch.nn.Linear(3, 4)

sequential = torch.nn.Sequential(layer1, layer2)

input_tensor = torch.ones((5, 2))
output_tensor = sequential(input_tensor)

print(input_tensor, output_tensor, input_tensor.shape, output_tensor.shape)

Build your MLP within the following torch.nn.Module object.

In [ ]:
########### Code #############
class PerovMLP(torch.nn.Module):
    def __init__(self, in_features=None, layers=None, activation=None):
        # You may either modify the above __init__ call to either take in hyperparameters as keyword arguments,
        # or hard-code them below. If you want to do hyperparameter search, then modifying to take in
        # keyword arguments is recommended.
        super().__init__()
        ########### Code #############

        # Implement your code here
        self.layers = []





        
        self.model = ...
        ########### Code #############



        
    def forward(self, x):
        x = self.model(x)

        return x

########### Code #############

## 2.3 <span style="color:blue">(10 points) </span>  Implement functions for training and testing

Now with a defined model architecture and DataLoaders, we will set up the training. We initialize a model instance, `model`, and then an `optimizer` to perform backpropagation with a variant of stochastic gradient descent. Two options are stochastic gradient descent (`torch.optim.SGD`) or the Adam optimizer (`torch.optim.Adam`). 



**Task 1:** Construct the optimizer by using the Adam optimizer, and pass the following as arguments: `model.parameters()`; a learning rate `lr` of `1e-3`; and L2 regularization with `weight_decay=0.01`. Then, define your loss function by using `MSE_loss()` to use mean squared error loss (important hint: make sure your input vectors to this call are the same shape; consider using `.view_as()`, `.flatten()`, or `.reshape()`).

Define your model, device, and optimizer. Use an L2 weight of 0.01.

In [ ]:
########### Code #############

# device to train on
device = 'cuda:0'
# define your model
model = PerovMLP(in_features=146).to(device)

# define your optimizer
optimizer = # Fill in 

# define your loss function
loss_fn = # Fill in 

# define number of epochs
epochs = 250

########### Code #############

**Task 2:** We have provided the skeleton code for training and validation loops, which is quite standardized for any training task. We use the `DataLoader` to loop over the training data's minibatches and use the model to predict the $E_{hull}$ probabilities for each minibatch. Use the loss initialized above to compute a loss on that minibatch; this requires both your model output and the ground-truth values. PyTorch can then compute your gradients with `loss.backward()`. Model weights are then updated by calling `optimizer.step()`. Clear the gradients from the previous calculation (minibatch) by calling `optimizer.zero_grad()` at the start of each minibatch processing. Complete the `train()` function following these guidelines. Similarly, complete the `validate()` function, which uses the validation `DataLoader` and computes the loss on the validation set, though note: *no gradients need be computed or model weights changed*. 


The `train()` and `validate()` functions implemented will operate on only one epoch, and should ultimately return a train loss and validation loss averaged over all minibatches in that one epoch.

In [ ]:
########### Code #############

def train(model, dataloader, optimizer, loss_fn, device):

    '''
    A function train on the entire dataset for one epoch.

    Args:
        model (torch.nn.Module): your model from before
        dataloader (torch.utils.data.DataLoader): DataLoader object for the train data
        optimizer (torch.optim.Optimizer(()): optimizer object to interface gradient calculation and optimization
        device (str): Your device (usually 'cuda:0' for your GPU)

    Returns:
        float: loss averaged over all the batches
    '''

    epoch_loss = []
    model.train() # Set model to training mode

    for batch in dataloader:
        X, y = batch
        X = X.to(device)
        y = y.to(device)

        # train your model on each batch here
        y_pred = model(X)

        ########### Code #############



        ########### Code #############

    return epoch_mean



def validate(model, dataloader, loss_fn, device):

    '''
    A function validate on the validation dataset for one epoch.

    Args:
        model (torch.nn.Module): your model for before
        dataloader (torch.utils.data.DataLoader): DataLoader object for the validation data
        device (str): Your device (usually 'cuda:0' for your GPU)

    Returns:
        float: loss averaged over all the batches

    '''

    val_loss = []
    model.eval() # Set model to evaluation mode
    with torch.no_grad():
        for batch in dataloader:
            X, y = batch
            X = X.to(device)
            y = y.to(device)

            # validate your model on each batch here
            y_pred = model(X)

            ########### Code #############




            ########### Code #############


    return epoch_mean
########### Code #############

## 2.4 <span style="color:blue">(5 points) </span>  Train and validate your model.

For regression tasks, one metric we can use to evaluate model performance is the coefficient of determination, or the $R^2$ score. It is defined as: 

\begin{equation}
    R^2 = 1 - \frac{ \sum_i^{N_\text{data}} (y_i - \hat f(\mathbf{x}_i))^2}{ \sum_i^{N_\text{data}} (y_i - \operatorname{mean}(y_i) )}
\end{equation}

where $i$ is the index for each sample, $y_i$ is the target value, $\hat f$ is the model, and $\mathbf{x}_i$ is the feature vector. The larger the $R^2$ score, the more accurate the prediction is. If your prediction is perfect $R^2 = 1$. Note that metrics like mean absolute error (MAE) or mean squared error (MSE) can provide more meaningful and interpretable measures of performance.



**Task:** After ensuring you've initialized your model and optimizer with the hyperparameters listed in Table 2, train and validate your model for 250 epochs. Record the average train and validation loss for each epoch and plot these on a single graph (the plotting code is provided). Then, report the *test* $R^2$ of your trained model on the test data^1, and visualize your prediction for train and test data with a scatter plot. Finally, briefly comment on the `hidden_layers_sizes` values as they pertain to the MLP's intercepts and coefficients. 

> **Hint**: You'll need to convert your prediction from `torch.Tensor` to `np.array` with `.cpu().detach().numpy()`, then compute the $R^2$ score with `sklearn.metrics.r2_score`. 

<div align="center">Table 2: Hyperparameters to be used in sections 2.4-2.6</div>

| `hidden_layer_sizes` | (256, 256, 256) |
| :--- | :--- |
| `activation` | nn.ReLU |
| `alpha` | 0.01 |
| `solver` | Adam |
| `early_stopping` | False |
| `epochs` | 250 |



In [ ]:
val_loss_curve = []
train_loss_curve = []

def update(progress_bar, train_loss, val_loss):
    progress_bar.set_postfix({"train_loss": train_loss, "val_loss": val_loss})

progress_bar = tqdm(range(epochs)) # this wraps your iteration in a handy progress bar to track any metrics.

for epoch in progress_bar:

    # Train your model on training data
    train_loss = train(model, train_dataloader, optimizer, loss_fn=loss_fn, device=device)

    # Validate your model on validation data
    val_loss = validate(model, val_dataloader, loss_fn=loss_fn, device=device)

    # Record train and loss performance
    train_loss_curve.append(train_loss)
    val_loss_curve.append(val_loss)

    update(progress_bar, train_loss, val_loss)


In [ ]:
plt.plot(train_loss_curve, label="training")
plt.plot(val_loss_curve, label="validation")
plt.legend()

In [ ]:
########### Code #############
from sklearn.metrics import r2_score

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))

yhat_train = # Fill in 
yhat_val = # Fill in 

yhat_train = yhat_train.cpu().detach().numpy()
yhat_val = yhat_val.cpu().detach().numpy()

ax[0].scatter(yhat_train, y_subtrain, label='Train', alpha=0.5)
ax[1].scatter(yhat_val, y_val, label='Test', alpha=0.5, c='orange')

ax[0].set_ylabel("True $E_{hull}$ (eV/atom)")
ax[0].set_xlabel("Predicted $E_{hull}$ (eV/atom)")
ax[1].set_xlabel("Predicted $E_{hull}$ (eV/atom)")
ax[0].set_title('Train')
ax[1].set_title('Validation')
fig.suptitle('Multi-Layer Perceptron')

print("Multi-Layer Perceptron training R^2 score: {:.2f}".format(r2_score(y_subtrain, yhat_train)))
print("Multi-Layer Perceptron validation R^2 score: {:.2f}".format((r2_score(y_val, yhat_val))))

########### Code #############

## 2.5 <span style="color:orange">(5 points) (grad) </span> Calculate model size

**Task**: Calculate the number of parameters used in your MLP given the provided `hidden_layer_sizes`. You can do this manually or iterate over `model.parameters()`. What does the input `hidden_layers_sizes = (256, 256, 256)` mean?

In [ ]:
########## Answer ############


########## Answer ############

**Write answer here**

## 2.6 <span style="color:blue">(5 points) </span>Chemical Transferability of One-Hot Representations

We created a holdout set, which includes elements in new positions not seen in the training set, to test your model's performance. 



**Task:** Load the test set (if you haven't already) and featurize the data with the label encoder from section 2.1. Validate your MLP by computing the $R^2$ score, and visualize your predictions with a scatter plot. Briefly describe your observations: does your model generalize well to this new data?

In [ ]:
########### Code #############

# Load the test dataset which contains elements not seen in the training data
perov_test = pd.read_csv("perov_val.csv")

# Your code to preprocess data and predict on this test dataset, including a scatterplot of predictions





print("MLP validation R^2 score: {:.2f}".format(r2_score(y_test, yhat_test)))
########### Code #############

Comment on your validation results and briefly explain.

**Write answer here**

## 2.7 <span style="color:orange">(10 points) (grad) </span> Featurize perovskites with physical descriptors

Let's try using a more informative featurization, with physical descriptors of the elements, which should contain more helpful information than a one-hot encoding.

**Task:** Load the data for atomic properties stored in `mendeleev.csv` with provided code. Use all the numerical features to construct a new feature matrix (please exclude atomic symbols which are strings). Split the original training data into a training (80%) set and a validation (20%) set; since we have continuous features again, rescale with `preprocessing.StandardScalar` as seen in Problem 1. Train on the training set with a MLP with the same hyperparameters we provided to you in section 2.6 (you will need to tweak the number of input features). Plot a scatterplot of predictions on the train and validation datasets.

In [ ]:
########### Code #############
elements_pd = pd.read_csv("mendeleev.csv")
elements_pd = elements_pd.set_index('symbol')







########### Code #############

In [ ]:
########### Code #############

# New dataloaders needed
new_train_data = PerovskiteDataset(X_atom_train, y_atom_train)
new_train_dataloader = DataLoader(new_train_data, batch_size=batch_size, shuffle=True)

new_val_data = PerovskiteDataset(X_atom_val, y_atom_val)
new_val_dataloader = DataLoader(new_val_data, batch_size=batch_size, shuffle=True)

# Fill in the rest





print("Retrained MLP Training R^2 score: {:.2f}".format(r2_score(y_atom_train, yhat_new_train)))
print("Retrained MLP Validation R^2 score: {:.2f}".format(r2_score(y_atom_val, yhat_new_val)))

########### Code #############

## 2.8 <span style="color:orange">(10 points) (grad) </span> Chemical transferability of Physical Descriptors

**Task**: Report the $R^2$ score on your test set using the model trained in part 2.8 and visualize your prediction with a scatter plot. Did your holdout predictions improve? Briefly explain why.

In [ ]:
########### Code #############


print("Retrained MLP Testing R^2 score: {:.2f}".format(r2_score(y_test, yhat_test)))

########### Code #############

Briefly comment on your validation and explain why.

**Write answer here**

---

# Submission

Congratulations! You've reached the end of the pset. Please submit your completed work as a `.ipynb` to Gradescope. Furthermore, if you had any AI-based assistance or worked with collaborators, please list them in the following cell.

**For submission:**
If your file is less than 10mb, feel free to turn in the `.ipynb` directly.

However, if it is >10mb, please take a look at the `# --- Configuration ---` section of the next cell. You will need to change the `NOTEBOOK_NAME` to match the name of your google colab notebook. After making the applicable changes, make sure to save your file, and then please run the cell which will reduce the size of your generated images and aim for a file size of less than 10 mb. Note that it will also delete cells tagged as "background". Check the output to ensure it didn't delete any of your outputs.

When successful, you'll be prompted to download the reformatted notebook which you can then upload to gradescope.

In [ ]:
import io
import os
import base64
import nbformat
import sys
import time
from PIL import Image
from IPython.display import display, Javascript, HTML

# --- Configuration ---
NOTEBOOK_NAME = "ASSIGNMENT_NAME.ipynb" # The name of your file
OUTPUT_FILENAME = "pset_1_nonbio_submission.ipynb"
TAG_TO_REMOVE = "background"
MAX_IMG_WIDTH = 800
# Standard locations where Colab saves notebooks.
# IF YOU CHANGE THE LOCATION OF YOUR NOTEBOOK PLEASE ADD THE PATH HERE.
COLAB_PATHS = [
    f"/content/drive/MyDrive/Colab Notebooks/{NOTEBOOK_NAME}",
    f"/content/drive/MyDrive/{NOTEBOOK_NAME}"
]
# ---------------------

def get_input_path():
    """Determines the path of the notebook based on the environment."""
    if 'google.colab' in sys.modules:
        from google.colab import drive
        # 1. Mount Drive
        if not os.path.exists('/content/drive'):
            print("Mounting Google Drive to access the notebook file...")
            drive.mount('/content/drive')
        
        for path in COLAB_PATHS:
            if os.path.exists(path):
                return path
        
        # Fallback if not found
        print(f"\nERROR: Could not find '{NOTEBOOK_NAME}' in your Google Drive.")
        print("Please ensure the file is saved in 'My Drive' or 'Colab Notebooks'.")
        return None
        
    else:
        # Local Jupyter (runs in current directory)
        return NOTEBOOK_NAME

def resize_base64_image(b64_str, mime_type):
    # (Same resize logic as before - keeping it brief for readability)
    try:
        img_data = base64.b64decode(b64_str)
        img = Image.open(io.BytesIO(img_data))
        if img.width > MAX_IMG_WIDTH:
            ratio = MAX_IMG_WIDTH / img.width
            new_height = int(img.height * ratio)
            img = img.resize((MAX_IMG_WIDTH, new_height), Image.Resampling.LANCZOS)
            buf = io.BytesIO()
            fmt = 'PNG' if 'png' in mime_type else 'JPEG'
            img.save(buf, format=fmt, optimize=True)
            return base64.b64encode(buf.getvalue()).decode('utf-8')
        return b64_str
    except Exception as e:
        return b64_str

def generate_submission():
    # Trigger a save in the browser
    display(Javascript('IPython.notebook.save_checkpoint();'))
    time.sleep(5)

    input_path = get_input_path()
    if not input_path:
        return

    print(f"Reading notebook from: {input_path}")

    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            nb = nbformat.read(f, as_version=4)
    except Exception as e:
        print(f"Error reading file: {e}")
        return

    new_cells = []

    # Filter Cells & Process Images
    for cell in nb.cells:
        tags = cell.get('metadata', {}).get('tags', [])
        tags = [t.lower() for t in tags] if tags else []
        if TAG_TO_REMOVE in tags:
            continue

        if 'outputs' in cell:
            for output in cell['outputs']:
                data = output.get('data', {})
                for mime_type in ['image/png', 'image/jpeg']:
                    if mime_type in data:
                        data[mime_type] = resize_base64_image(data[mime_type], mime_type)
        new_cells.append(cell)

    nb.cells = new_cells

    with open(OUTPUT_FILENAME, 'w', encoding='utf-8') as f:
        nbformat.write(nb, f)

    # Download logic for Colab
    if 'google.colab' in sys.modules:
        from google.colab import files
        print(f"Downloading {OUTPUT_FILENAME}...")
        files.download(OUTPUT_FILENAME)
        print(f"Success! {OUTPUT_FILENAME} downloaded.")
    else:
        print(f"Success! {OUTPUT_FILENAME} created.")
        display(HTML(f'<br/><a href="{OUTPUT_FILENAME}" download><b>Click here to download {OUTPUT_FILENAME}</b></a>'))

generate_submission()